In [0]:
# ==================================================
# BRONZE LAYER – INCREMENTAL INGESTION
# Source Volume: bronze_landing_zone
# Metadata Volume: bronze_metadata
# ==================================================

from pyspark.sql.functions import current_timestamp, col

# --------------------------------------------------
# 1. Paths (do volumes)
# --------------------------------------------------
SOURCE_VOLUME = "/Volumes/workspace/default/bronze_landing_zone/"
METADATA_VOLUME = "/Volumes/workspace/default/bronze_metadata/"

PRICING_SOURCE = SOURCE_VOLUME + "source_pricing/"
QUICK_SOURCE   = SOURCE_VOLUME + "source_quick/"

PRICING_CHECKPOINT = METADATA_VOLUME + "pricing/checkpoint"
PRICING_SCHEMA     = METADATA_VOLUME + "pricing/schema"
QUICK_CHECKPOINT   = METADATA_VOLUME + "quick/checkpoint"
QUICK_SCHEMA       = METADATA_VOLUME + "quick/schema"

# Ensure directories exist (idempotent – does nothing if already exists)
dbutils.fs.mkdirs(PRICING_SOURCE)
dbutils.fs.mkdirs(QUICK_SOURCE)
dbutils.fs.mkdirs(PRICING_CHECKPOINT)
dbutils.fs.mkdirs(PRICING_SCHEMA)
dbutils.fs.mkdirs(QUICK_CHECKPOINT)
dbutils.fs.mkdirs(QUICK_SCHEMA)

# Table names
BRONZE_PRICING_TABLE = "bronze_pricing"
BRONZE_QUICK_TABLE   = "bronze_quick"


# --------------------------------------------------
# 2. Ingest Pricing Data
# --------------------------------------------------
df_pricing = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", PRICING_SCHEMA)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true")
    .load(PRICING_SOURCE)
)

df_pricing = df_pricing \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .withColumn("_bronze_ingested_at", current_timestamp())

query_pricing = (df_pricing.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", PRICING_CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_PRICING_TABLE)
)
query_pricing.awaitTermination()
print("✅ Pricing ingestion complete")

# --------------------------------------------------
# 3. Ingest Quick Orders Data
# --------------------------------------------------
df_quick = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", QUICK_SCHEMA)
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("header", "true")
    .load(QUICK_SOURCE)
)

df_quick = df_quick \
    .withColumn("_source_file", col("_metadata.file_path")) \
    .withColumn("_bronze_ingested_at", current_timestamp())

query_quick = (df_quick.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", QUICK_CHECKPOINT)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable(BRONZE_QUICK_TABLE)
)
query_quick.awaitTermination()
print("✅ Quick orders ingestion complete")

# --------------------------------------------------
# 4. Verify
# --------------------------------------------------
print(f"Pricing table rows: {spark.table(BRONZE_PRICING_TABLE).count()}")
print(f"Quick table rows:   {spark.table(BRONZE_QUICK_TABLE).count()}")

✅ Pricing ingestion complete
✅ Quick orders ingestion complete
Pricing table rows: 100000
Quick table rows:   30600
